# Simple Bot: Integrating LLMs into LangGraph

### Objectives:
1. **Define State Structure**: Maintain a conversation history using a list of messages (such as `HumanMessage`, `AIMessage`, etc.) with LangGraph's message reducer.
2. **Initialize GPT-4o Model**: Set up `ChatOpenAI(model="gpt-4o")` using `langchain-openai`.
3. **Sending and Handling Different Message Types**: Work with `SystemMessage`, `HumanMessage`, and `AIMessage`.
4. **Building and Compiling the Graph**: Create a runnable graph agent with a chatbot node connected from `START` to `END`.

**Main Goal:** Understand how LLMs are integrated cleanly as nodes in LangGraph workflows.

## 0. Prerequisites & Imports

In [ ]:
# Install dependencies if needed:
# !pip install langgraph langchain-core langchain-openai python-dotenv

In [ ]:
import os
from typing import Annotated, Sequence, TypedDict
from dotenv import load_dotenv

# LangChain Message Types
from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    AIMessage,
    SystemMessage
)
from langchain_openai import ChatOpenAI

# LangGraph Components
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

# Load environment variables (OPENAI_API_KEY)
load_dotenv()

## 1. Objective 1: Define State Structure with Messages

In LangGraph, conversation states hold a list of messages.
Using `Annotated[Sequence[BaseMessage], add_messages]` ensures that new messages returned by nodes are **appended** to the state history rather than replacing it.

In [ ]:
# 1. Define State Structure
class BotState(TypedDict):
    # add_messages is the built-in reducer that appends messages
    messages: Annotated[Sequence[BaseMessage], add_messages]

## 2. Objective 2: Initialize GPT-4o Model

We initialize LangChain's `ChatOpenAI` targeting `gpt-4o`.

In [ ]:
# Set or check your OpenAI API Key
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"

# Initialize the GPT-4o model
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.7
)

print("LLM Initialized:", llm.model_name)

## 3. Objective 3: Handling Different Message Types in a Node

LangChain supports several message classes:
- **`SystemMessage`**: Sets instructions and persona for the bot.
- **`HumanMessage`**: Represents user input.
- **`AIMessage`**: Represents responses generated by the model.

Inside the graph node, we pass the messages list directly to `llm.invoke(state["messages"])` and return the response wrapped in a dictionary under `"messages"`.

In [ ]:
# Node that invokes the LLM
def chatbot_node(state: BotState) -> dict:
    """
    Receives the current state with conversation history,
    calls the LLM, and returns the generated AIMessage.
    """
    messages = state["messages"]
    response = llm.invoke(messages)
    return {"messages": [response]}

## 4. Objective 4: Building and Compiling the Graph

We assemble the workflow:
1. `StateGraph(BotState)`
2. Add `chatbot_node`
3. Connect `START -> chatbot_node -> END`
4. Compile the graph with `workflow.compile()`

In [ ]:
# 1. Initialize Graph with State
workflow = StateGraph(BotState)

# 2. Add Node
workflow.add_node("chatbot", chatbot_node)

# 3. Add Edges
workflow.add_edge(START, "chatbot")
workflow.add_edge("chatbot", END)

# 4. Compile Graph
app = workflow.compile()
print("Simple Bot Graph compiled successfully!")

## 5. Main Goal: Executing the Graph with LLM Integration

Let's test sending both `SystemMessage` (defining behavior) and `HumanMessage` (asking questions).

In [ ]:
# Test 1: Sending SystemMessage and HumanMessage
input_messages = [
    SystemMessage(content="You are a helpful and concise AI programming tutor specialized in LangGraph."),
    HumanMessage(content="In one sentence, why do we use LangGraph?")
]

# Invoke the Graph (requires valid OPENAI_API_KEY)
try:
    output = app.invoke({"messages": input_messages})
    last_message = output["messages"][-1]
    print("AI Response:")
    print(last_message.content)
except Exception as e:
    print(f"Note: Set OPENAI_API_KEY in your environment to execute live API calls.\nError details: {e}")

## 6. Multi-Turn Conversation Example

Demonstrating handling a full sequence of message types (`SystemMessage`, `HumanMessage`, `AIMessage`, and subsequent `HumanMessage`).

In [ ]:
# Multi-turn conversation demonstration
conversation_history = [
    SystemMessage(content="You are a friendly pirate assistant."),
    HumanMessage(content="Hello, what is your name?"),
    AIMessage(content="Ahoy matey! They call me Captain Code!"),
    HumanMessage(content="What are we building today?")
]

print("--- Message History Handled by Graph ---")
for msg in conversation_history:
    print(f"{msg.__class__.__name__}: {msg.content}")

try:
    multi_turn_output = app.invoke({"messages": conversation_history})
    print("\n--- New AI Reply ---")
    print(multi_turn_output["messages"][-1].content)
except Exception as e:
    print(f"\n(Live API call skipped: {e})")